# Shapley Value-Based Client Selection (S-FedAvg)

Implements the **S-FedAvg** algorithm from:
> *Game of Gradients: Mitigating Irrelevant Clients in Federated Learning*
> Nagalapatti & Narayanam, AAAI 2021

### Algorithm Overview
1. All clients compute gradient updates locally
2. Server estimates each client's **Shapley value** using a held-out validation set
3. Shapley values → softmax probabilities → probabilistic client selection
4. Only selected clients' updates are aggregated via FedAvg

In [ ]:
import random
import copy
import torch
import numpy as np


def _apply_update(global_model, update):
    """
    Apply a single client's gradient update to a deep copy of global_model.
    Returns the updated model (does not modify global_model in-place).
    """
    updated_model = copy.deepcopy(global_model)
    new_state = {}
    for key in global_model.state_dict():
        new_state[key] = global_model.state_dict()[key] + update[key].float()
    updated_model.load_state_dict(new_state)
    return updated_model


def _apply_updates_subset(global_model, client_ids, client_updates):
    """
    Apply updates from a *subset* of clients via FedAvg and return the resulting model.
    """
    if len(client_ids) == 0:
        return copy.deepcopy(global_model)

    updated_model = copy.deepcopy(global_model)
    new_state = {}
    for key in global_model.state_dict():
        avg_update = torch.mean(
            torch.stack([client_updates[c][key].float() for c in client_ids]), dim=0
        )
        new_state[key] = global_model.state_dict()[key] + avg_update

    updated_model.load_state_dict(new_state)
    return updated_model


def _val_accuracy(model, val_loader, device):
    """
    Evaluate model accuracy on the validation DataLoader.
    This is the characteristic function v(S) used in Shapley computation.
    """
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0


def compute_shapley_values(global_model, client_updates, val_loader, device, T=10):
    """
    Compute Shapley values for each client using Truncated Monte Carlo estimation.

    This follows Algorithm 1 of the S-FedAvg paper:
      - For T random permutations of clients:
          - Incrementally add each client's update via FedAvg
          - Measure marginal improvement on the validation set
          - Accumulate marginal contributions
      - Average across all permutations

    Args:
        global_model  : The current global model (PyTorch nn.Module)
        client_updates: dict {client_id: state_dict_of_update}
        val_loader    : DataLoader for the server's validation set
        device        : 'cuda' or 'cpu'
        T             : Number of permutation samples (higher = more accurate, slower)

    Returns:
        shapley_scores: dict {client_id: float}
    """
    client_ids = list(client_updates.keys())
    n = len(client_ids)
    shapley_scores = {c: 0.0 for c in client_ids}

    # Baseline: utility of global model with NO updates applied
    baseline_score = _val_accuracy(global_model, val_loader, device)

    for _ in range(T):
        perm = client_ids.copy()
        random.shuffle(perm)

        prev_score = baseline_score
        coalition = []  # incrementally growing coalition

        for c in perm:
            coalition.append(c)
            # Apply all updates in current coalition via FedAvg
            temp_model = _apply_updates_subset(global_model, coalition, client_updates)
            new_score = _val_accuracy(temp_model, val_loader, device)

            # Marginal contribution of client c
            shapley_scores[c] += (new_score - prev_score)
            prev_score = new_score

    # Average across T permutations
    for c in client_ids:
        shapley_scores[c] /= T

    return shapley_scores


def sfedavg_selection(shapley_scores, k):
    """
    Select k clients probabilistically using softmax over Shapley scores.

    As described in the paper:
        P_c = softmax(phi)   where phi = shapley_scores
        S^{t+1} ~ Multinomial(K, P_c)  (sample k clients without replacement)

    Args:
        shapley_scores: dict {client_id: float}
        k             : Number of clients to select

    Returns:
        selected: list of selected client IDs
    """
    client_ids = list(shapley_scores.keys())
    scores = np.array([shapley_scores[c] for c in client_ids], dtype=np.float64)

    # Stable softmax (subtract max for numerical stability)
    scores -= scores.max()
    exp_scores = np.exp(scores)
    probabilities = exp_scores / exp_scores.sum()

    # Sample k clients without replacement
    k = min(k, len(client_ids))  # guard against k > available clients
    selected_indices = np.random.choice(
        len(client_ids), size=k, replace=False, p=probabilities
    )
    selected = [client_ids[i] for i in selected_indices]
    return selected
